# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# .env file is in module_3/.env with:
# OPENAI_API_KEY, CHROMA_OPENAI_API_KEY, TAVILY_API_KEY

In [4]:
load_dotenv()

True

### VectorDB Instance

In [5]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
)

In [7]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game],
    )

print(f"Indexed {collection.count()} games into the 'udaplay' collection.")

Indexed 15 games into the 'udaplay' collection.


In [9]:
query = "When was Pokémon Gold and Silver released?"
results = collection.query(query_texts=[query], n_results=3)

print(f"Query: {query}\n")
for i, (metadata, distance) in enumerate(
    zip(results["metadatas"][0], results["distances"][0]), start=1
):
    print(
        f"{i}. {metadata['Name']} ({metadata['Platform']}, {metadata['YearOfRelease']}) "
        f"[distance: {distance:.4f}]"
    )

Query: When was Pokémon Gold and Silver released?

1. Pokémon Gold and Silver (Game Boy Color, 1999) [distance: 0.1127]
2. Pokémon Ruby and Sapphire (Game Boy Advance, 2002) [distance: 0.1467]
3. Super Mario 64 (Nintendo 64, 1996) [distance: 0.2207]
